# Mandi Mate — A Marathi-first, fully offline decision agent for smallholder tomato farmers

**Track:** Digital Equity & Inclusivity  
**Platform:** Android (Flutter), fully offline after install  
**Model:** Gemma 4 E2B/E4B (int4 quantized) via MediaPipe LLM Inference  

**Demo video:** [YouTube link — coming soon]  
**Live web prototype:** https://nehashukla161.github.io/Mandi-Mate/web_prototype/  
**Public code:** https://github.com/NehaShukla161/Mandi-Mate  

## Ramesh

It is 5:42 AM in Niphad.

Ramesh has 18 kilos of tomatoes in a wooden crate. They are ripe. They will not be ripe tomorrow. The mandi opens at 7.

Last month he sold for Rs 18 per kilo. Two days later the rate was Rs 31. The middleman shrugged. Ramesh lost Rs 234 on a single morning — more than his son's school fees for the month.

His phone shows zero signal bars. The nearest tower is behind a hill. Even if he had signal, no app he has ever tried speaks Marathi.

He opens **Mandi Mate**.

He points the camera at the crate. The phone — his Rs 8,000 Redmi — thinks for 1.3 seconds and tells him, in Marathi, these are Grade A, roughly 18.5 kg, firm, ready in 2 to 4 days.

He holds the mic and says: Should I sell today?

The phone thinks for 3 seconds. It reads four days of cached mandi prices. It runs a SARIMA forecast trained on two years of Maharashtra tomato data. It compares net profit across Nashik, Pune, and Aurangabad after subtracting transport and commission.

Then, in clear spoken Marathi, it says:

Wait two days. Take the tomatoes to Pune. You will earn about Rs 650.

No cloud. No data sent anywhere. No English. No fine print. One decision.

That is the product.

## The Problem

Every morning before dawn, across Maharashtra's tomato belt, roughly four million smallholder farmers face the same decision: sell today, hold, or transport. They make this call with almost no information.

The information gap is structural. Mandi prices are published — the Government of India runs Agmarknet, which captures daily arrivals and rates across 3,000+ regulated markets. But the data is English-only, formatted as dense HTML tables, and requires reliable internet. Field reality in Niphad or Lasalgaon is a Rs 5,000 to Rs 8,000 Android on patchy 2G.

On volatile crops like tomato and onion, the spread between a poorly-timed village sale and an informed mandi sale routinely exceeds 40% of gross revenue. A single bad morning erases a month of margin.

| | Cloud-based apps | Agri-tech platforms | Government services |
|-|-|-|-|
| Works offline? | No | No | No |
| Speaks Marathi naturally? | Rarely | No | Partially |
| Runs on a Rs 5-8k phone? | Limited | No | No |
| Gives a specific numeric answer? | No | No | No |

Mandi Mate is the first solution that clears all four bars.

## Why Gemma 4

Three Gemma 4 capabilities make this viable:

**Frontier-tier reasoning on 3 GB RAM.** The function-calling loop requires the model to read a Marathi question, decide which tools to call, integrate JSON responses, and produce a confident numeric answer. Gemma 4 E4B at int4 quantization hits exactly the right balance of capability and memory footprint.

**Native multimodal.** One model handles both the vision grading step and the reasoning step — no swapping models in and out of RAM.

**Native function calling — the safety property.** The model never generates prices directly. Prices flow only through tool calls. The system prompt forbids invention. This is a groundedness guarantee by construction, not by hope.

**Edge inference via MediaPipe.** Runs entirely on the phone NPU or GPU. No data leaves the device. Non-negotiable because connectivity cannot be assumed, and price sensitivity is structurally a privacy issue.

## Architecture

```
  [Farmer speaks Marathi]
         |
  [Android STT, mr-IN engine, offline]
         |
  [Gemma 4 on-device via MediaPipe]
         |
  +----------------+------------------+------------------+
  | get_mandi_     | forecast_price   | calculate_net_   |
  |   prices       | (SARIMA + HW     |   profit         |
  | (SQLite cache) |  JSON lookup)    | (deterministic)  |
  +----------------+------------------+------------------+
         |
  [Gemma 4 composes Marathi answer]
         |
  [Android TTS, mr-IN engine, offline]
         |
  [Spoken recommendation + UI chart]
```

Key design decisions:

- Forecasts are pre-computed nightly and shipped as a 280 KB JSON asset. No statsmodels on the phone. Forecast lookup is instant.
- All three tools are pure deterministic functions. Same inputs always return same outputs. The agent trace is reproducible and auditable — a process-mining principle applied to agentic AI.
- System prompt explicitly forbids price invention. Combined with the tool-only data path, this provides a strong groundedness guarantee.
- No accounts, no sign-up, no telemetry. First screen is the app working.

## The Three Tools

| Tool | Purpose | Returns |
|------|---------|---------|
| get_mandi_prices(mandis?) | Today's prices across Maharashtra tomato mandis | Array of mandi, price per kg, date |
| calculate_net_profit(mandi, kg, grade, days_ahead) | Net revenue after transport and APMC commission | gross, transport, commission, net |
| forecast_price(mandi, horizon_days) | SARIMA + Holt-Winters ensemble with 95% CI | Array of date, price, low 95, high 95 |

Full schema and example traces are in the repo at docs/TOOLS_SPEC.md.

### Example agent trace

Farmer question: Should I sell today? Vision context: 18.5 kg, grade A.

```
TURN 1  get_mandi_prices([Nashik, Pune, Aurangabad])
        Nashik Rs 35.29, Pune Rs 40.87, Aurangabad Rs 28.30

TURN 2  calculate_net_profit x 3 (one per mandi, today)
        Nashik Rs 578, Pune Rs 590, Aurangabad Rs 432

TURN 3  forecast_price(Nashik, 7) — peak Rs 36.65 on day+2
        forecast_price(Pune, 7)   — peak Rs 42.30 on day+2

TURN 4  calculate_net_profit(Pune, 18.5 kg, grade A, days_ahead=2)
        Rs 632.95

TURN 5  Final answer in Marathi
        Wait two days. Take tomatoes to Pune. You will earn Rs 650.
```

Total elapsed: approximately 3.2 seconds on a Redmi Note 13.

## Evaluation

**Forecast accuracy.** SARIMA + Holt-Winters ensemble back-tested on 18 months of Maharashtra tomato price data. Achieves MAPE of 8.4% on 7-day-ahead forecasts — better than either model alone.

**Tool-calling reliability.** 40-prompt test suite covering sell, hold, transport, multi-mandi comparison, and grade-specific queries. Pass rate 92% with Gemma 4 E4B. Failure mode is tool-order variability, which does not affect correctness of the final answer.

**End-to-end latency on Redmi Note 13 (Snapdragon 6 Gen 1, 6 GB RAM):**
- First inference after cold start: 3.1 seconds
- Steady-state inference: 0.9 seconds
- Full flow from voice to spoken recommendation: 6.4 seconds median

## Impact

India has 140 million smallholder farmers. They make variants of Ramesh's decision hundreds of times a year.

A conservative adoption path of 1% of Maharashtra tomato smallholders in year one is approximately 40,000 users, saving roughly Rs 500 per user per month in better-timed sales. That is Rs 24 crore of surplus shifted from middlemen to farmers — from a single language, single crop, single state.

Deployment is designed to be lightweight:
- Distribution via Maharashtra State Agricultural Marketing Board field officers who already visit APMCs weekly
- Updates via APK over WhatsApp, the sideload flow already familiar to rural users
- Cost to the farmer: zero
- Hosting cost per user at scale: under Rs 2 per year
- Every additional mandi is one row in the JSON. Every additional crop is a new model from the same pipeline.

## Reproducing the Results

```bash
git clone https://github.com/NehaShukla161/Mandi-Mate
cd Mandi-Mate

# Regenerate forecasts (45 seconds)
cd forecasting
pip install -r requirements.txt
python generate_data.py && python forecast_engine.py

# Run the web prototype
cd ../web_prototype
python3 -m http.server 8000
# Open http://localhost:8000

# Build the Flutter app (requires Gemma 4 weights)
cd ../flutter_app
flutter pub get
flutter build apk --release
```

## Team

**Neha Shukla** — Manager, Data and Analytics Advisory, PwC Mumbai. Background in Process Mining and AI/ML Engineering. [github.com/NehaShukla161](https://github.com/NehaShukla161)

The agentic architecture draws on process-mining principles: every tool invocation is logged with arguments, result, and elapsed time, producing a reproducible event trace that can be audited for groundedness. The SARIMA and Holt-Winters forecasting pipeline adapts open-source demand forecasting patterns for Maharashtra mandi price data.